In [9]:
import os, sys
os.environ['CUDA_VISIBLE_DEVICES'] = '2,3'

root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if root not in sys.path:
    sys.path.insert(0, root)

In [10]:
import numpy as np
import torch
import torch.nn as nn
import pandas as pd
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

from cvae.models import CVAESimpleEnc
from cvae.datasets import CVAEAllDataset
from cvae.utils import (
    CONDITION_LENGTH, MAX_FASTA_LENGTH, MAX_SEQ_LENGTH, ALPHABET, PAD_TOKEN_ID,
    finetune_collate_fn, vae_loss_fn_with_cond
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device, torch.cuda.device_count())

cuda 2


In [11]:
batch_size = 2048
df = pd.read_csv("../../dataset/merged_all.csv", keep_default_na=False, na_values=[''])
train_val_df, test_df = train_test_split(df, test_size=0.1, stratify=df['length'], random_state=42)
train_df, val_df = train_test_split(train_val_df, test_size=0.1, stratify=train_val_df['length'], random_state=42)

In [12]:
FEATURES = [
    "length",
    "is_assembled",
    "ap",
    "has_beta_sheet_content",
    "hydrophobic_moment",
    "net_charge",
]

In [13]:
train_dataset = CVAEAllDataset(train_df, max_fasta_length=MAX_FASTA_LENGTH, random_mask=True)
val_dataset   = CVAEAllDataset(val_df, max_fasta_length=MAX_FASTA_LENGTH)
test_dataset  = CVAEAllDataset(test_df, max_fasta_length=MAX_FASTA_LENGTH)

train_loader_ft = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=finetune_collate_fn)
val_loader_ft   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=finetune_collate_fn)
test_loader_ft  = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=finetune_collate_fn)

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device, torch.cuda.device_count())

cvae_model = CVAESimpleEnc(
    encoder_hidden_dim=256,
    num_encoder_layers=2,
    vocab_size=len(ALPHABET),
    latent_dim=24,
    cond_dim=CONDITION_LENGTH,
    max_seq_length=MAX_SEQ_LENGTH,
    decoder_hidden_dim=256,
    num_decoder_layers=2,
    nhead=8,
    dropout=0.1)

pretrained_state_dict = torch.load("../cvae/chkpts/finetuned_cvae.pt", map_location=device, weights_only=True)
cvae_model.load_state_dict(pretrained_state_dict)

cuda 2


<All keys matched successfully>

In [15]:
cvae_model.eval()

if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs")
    cvae_model = nn.DataParallel(cvae_model)

cvae_model.to(device)
print("CVAE initialized with finetuned architecture")

Using 2 GPUs
CVAE initialized with finetuned architecture


In [16]:
def evaluate(model, dataloader, pad_idx, device, kl_weight, cond_weight):
    model.eval()
    total = {
        "loss":    0.0,
        "recon":   0.0,
        "kl":      0.0,
        "mse":     0.0,
        "tokens":  0,
    }

    with torch.no_grad():
        for tokens, tgt_tokens, conds, mask in dataloader:
            B = tokens.size(0)
            tokens     = tokens.to(device)
            tgt_tokens = tgt_tokens.to(device)
            conds      = conds.to(device)
            mask       = mask.to(device)

            logits, mu, logvar, prior_mu, prior_logvar, bc_logit, cc_pred, mask_logit = \
                model(tokens, conds, mask)

            #vocab_size = logits.size(-1)

            loss, recon, kl, _ = vae_loss_fn_with_cond(
                logits=logits.view(-1, logits.size(-1)),
                tgt=tgt_tokens.view(-1),
                mu=mu, logvar=logvar, prior_mu=prior_mu, prior_logvar=prior_logvar,
                bc_logit=bc_logit, cc_pred=cc_pred, mask_logit=mask_logit,
                cond=conds, mask=mask,
                pad_idx=PAD_TOKEN_ID, kl_weight=kl_weight, lambda_bin=1.0, lambda_cont=1.0
            )

            total["loss"]  += loss.item() * B
            total["recon"] += recon.item() * B
            total["kl"]    += kl.item() * B
            total["tokens"] += (tgt_tokens != pad_idx).sum().item()

    N = len(dataloader.dataset)
    total["loss"]  /= N
    total["recon"] /= N
    total["kl"]    /= N

    total["ppl"] = float(torch.exp(torch.tensor(total["recon"] * total["tokens"] / total["tokens"])))

    return total


metrics = evaluate(
    model=cvae_model,
    dataloader=val_loader_ft,
    pad_idx=PAD_TOKEN_ID,
    device=device,
    kl_weight=0.02,
    cond_weight=50.0
)

print("==== Validation metrics ====")
print(f"Loss: {metrics['loss']:.4f} | Recon: {metrics['recon']:.4f} | KL: {metrics['kl']:.4f}")
print(f"Perplexity: {metrics['ppl']:.2f}")

/home/go46vuw/miniconda3/lib/python3.12/site-packages/torch/nn/modules/transformer.py:508: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /home/conda/feedstock_root/build_artifacts/libtorch_1744233415586/work/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


==== Validation metrics ====
Loss: 1.3955 | Recon: 1.0395 | KL: 17.7983
Perplexity: 2.83


In [17]:
metrics = evaluate(
    model=cvae_model,
    dataloader=test_loader_ft,
    pad_idx=PAD_TOKEN_ID,
    device=device,
    kl_weight=0.02,
    cond_weight=50.0
)

print("==== Test metrics ====")
print(f"Loss: {metrics['loss']:.4f} | Recon: {metrics['recon']:.4f} | KL: {metrics['kl']:.4f}")
print(f"Perplexity: {metrics['ppl']:.2f}")

==== Test metrics ====
Loss: 1.3940 | Recon: 1.0370 | KL: 17.8506
Perplexity: 2.82


In [18]:
from torch.utils.data import Subset, DataLoader

def make_complete_case_subset(dataset, cond_dim=CONDITION_LENGTH):
    """
    Returns a torch.utils.data.Subset containing only samples where all cond dims are observed.
    Assumes dataset __getitem__ returns (seq, cond, mask) or (seq, cond, mask, ...) as in your CVAEAllDataset.
    """
    idxs = []
    for i in range(len(dataset)):
        item = dataset[i]
        # support both (seq, cond, mask) and (seq, cond, mask, extra...)
        mask = item[2]
        if (mask.sum().item() == cond_dim):
            idxs.append(i)
    return Subset(dataset, idxs)

def apply_random_mask_on_top(mask, p_keep=0.5):
    """
    Randomly drop observed dims (mask==1) with probability 1-p_keep.
    Keeps truly-missing dims at 0.
    """
    # bernoulli keep mask for each element
    keep = (torch.rand_like(mask) < p_keep).float()
    return mask * keep

@torch.no_grad()
def evaluate_ppl(model, dataloader, pad_idx, device, kl_weight,
                 mask_mode="observed", p_keep=0.5):
    """
    mask_mode:
      - "observed": use dataset mask as-is
      - "full": force all ones (ONLY valid if cond values are truly present)
      - "random_on_top": apply random masking on top of observed mask
    Returns mean recon CE and PPL = exp(CE).
    """
    model.eval()
    total_loss = 0.0
    total_recon = 0.0
    total_kl = 0.0
    n_samples = 0

    for tokens, tgt_tokens, conds, mask in dataloader:
        B = tokens.size(0)
        tokens = tokens.to(device)
        tgt_tokens = tgt_tokens.to(device)
        conds = conds.to(device)
        mask = mask.to(device)

        if mask_mode == "observed":
            mask_eff = mask
        elif mask_mode == "full":
            mask_eff = torch.ones_like(mask)
        elif mask_mode == "random_on_top":
            mask_eff = apply_random_mask_on_top(mask, p_keep=p_keep)
        else:
            raise ValueError(f"Unknown mask_mode={mask_mode}")

        logits, mu, logvar, prior_mu, prior_logvar, bc_logit, cc_pred, mask_logit = \
            model(tokens, conds, mask_eff)

        loss, recon, kl, _ = vae_loss_fn_with_cond(
            logits=logits.view(-1, logits.size(-1)),
            tgt=tgt_tokens.view(-1),
            mu=mu, logvar=logvar, prior_mu=prior_mu, prior_logvar=prior_logvar,
            bc_logit=bc_logit, cc_pred=cc_pred, mask_logit=mask_logit,
            cond=conds, mask=mask_eff,
            pad_idx=pad_idx,
            kl_weight=kl_weight,
            lambda_bin=1.0, lambda_cont=1.0, label_smoothing=0.0
        )

        total_loss += loss.item() * B
        total_recon += recon.item() * B
        total_kl += kl.item() * B
        n_samples += B

    mean_loss = total_loss / n_samples
    mean_recon = total_recon / n_samples
    mean_kl = total_kl / n_samples
    ppl = float(torch.exp(torch.tensor(mean_recon)))  # recon is mean CE per token (ignore_index handles pads)

    return {"loss": mean_loss, "recon": mean_recon, "kl": mean_kl, "ppl": ppl}


test_complete = make_complete_case_subset(test_dataset, cond_dim=CONDITION_LENGTH)

test_loader_complete = DataLoader(
    test_complete, batch_size=batch_size, shuffle=False, collate_fn=finetune_collate_fn
)

# ---- run evals ----
m_obs = evaluate_ppl(cvae_model, test_loader_ft, PAD_TOKEN_ID, device, kl_weight=0.02, mask_mode="observed")
m_full = evaluate_ppl(cvae_model, test_loader_complete, PAD_TOKEN_ID, device, kl_weight=0.02, mask_mode="full")
m_rand = evaluate_ppl(cvae_model, test_loader_complete, PAD_TOKEN_ID, device, kl_weight=0.02, mask_mode="random_on_top", p_keep=0.5)

print("TEST (all) observed-mask PPL:", m_obs["ppl"])
print("TEST (complete-case) full-mask PPL:", m_full["ppl"])
print("TEST (complete-case) random-on-top PPL:", m_rand["ppl"])


TEST (all) observed-mask PPL: 1.6718480587005615
TEST (complete-case) full-mask PPL: 1.6336920261383057
TEST (complete-case) random-on-top PPL: 1.6874926090240479


In [19]:
import numpy as np
import torch
import torch.nn.functional as F

FEATURES = [
    "length",
    "is_assembled",
    "ap",
    "has_beta_sheet_content",
    "hydrophobic_moment",
    "net_charge",
]

IDX_BIN  = [1, 3]         # binary descriptors in your setup
IDX_CONT = [0, 2, 4, 5]   # continuous descriptors in your setup

@torch.no_grad()
def evaluate_aux_heads_per_dim(model, dataloader, device, max_pos_w=50.0, thresh=0.5):
    model.eval()

    # Accumulators
    mask_stats = {f: {"bce_sum": 0.0, "acc_sum": 0.0, "pos_sum": 0.0, "n_sum": 0.0} for f in FEATURES}

    bin_stats  = {FEATURES[i]: {"bce_sum": 0.0, "acc_sum": 0.0, "bal_acc_sum": 0.0,
                                "pos_sum": 0.0, "n_obs": 0.0, "tp": 0.0, "tn": 0.0, "fp": 0.0, "fn": 0.0}
                  for i in IDX_BIN}

    cont_stats = {FEATURES[i]: {"mae_sum": 0.0, "mse_sum": 0.0, "n_obs": 0.0, "obs_rate_sum": 0.0}
                  for i in IDX_CONT}

    total_n = 0

    for tokens, tgt_tokens, conds, mask in dataloader:
        B = tokens.size(0)
        total_n += B

        tokens = tokens.to(device)
        conds  = conds.to(device)
        mask   = mask.to(device)

        logits, mu, logvar, prior_mu, prior_logvar, bc_logit, cc_pred, mask_logit = \
            model(tokens, conds, mask)

        # -------------------------
        # Mask reconstruction head
        # -------------------------
        # Compute per-dimension BCE with logits and accuracy
        # Use pos_weight per dim (as you did) for BCE stability
        pos_frac = mask.mean(dim=0).clamp(1e-4, 1-1e-4)
        pos_w = ((1 - pos_frac) / pos_frac).clamp(1.0, max_pos_w)

        # per-dim BCE
        # BCEWithLogits returns mean over all elements if reduction="mean", so do it manually:
        # loss = - [ y*log(sigmoid(x)) + (1-y)*log(1-sigmoid(x)) ] with pos_weight
        # easiest: call BCEWithLogits with reduction="none"
        bce_per = F.binary_cross_entropy_with_logits(mask_logit, mask, pos_weight=pos_w, reduction="none")  # (B,D)
        bce_dim = bce_per.mean(dim=0)  # (D,)

        mask_pred = (torch.sigmoid(mask_logit) >= thresh).float()
        acc_dim = (mask_pred == mask).float().mean(dim=0)
        pos_dim = mask.float().mean(dim=0)

        for j, feat in enumerate(FEATURES):
            mask_stats[feat]["bce_sum"] += float(bce_dim[j]) * B
            mask_stats[feat]["acc_sum"] += float(acc_dim[j]) * B
            mask_stats[feat]["pos_sum"] += float(pos_dim[j]) * B
            mask_stats[feat]["n_sum"]   += B

        # -------------------------
        # Binary descriptor heads
        # -------------------------
        # bc_logit is (B, len(IDX_BIN)), each corresponds to FEATURES[IDX_BIN[j]]
        for head_j, feat_idx in enumerate(IDX_BIN):
            feat = FEATURES[feat_idx]
            m = mask[:, feat_idx].bool()
            if not m.any():
                continue

            tgt = conds[:, feat_idx][m].float()
            logit = bc_logit[:, head_j][m]
            pred = (torch.sigmoid(logit) >= thresh).float()

            # BCE (unweighted here; you can add weighting if you want, but for reporting keep it simple)
            bce = F.binary_cross_entropy_with_logits(logit, tgt, reduction="mean")

            # confusion matrix
            tp = ((pred == 1) & (tgt == 1)).sum().item()
            tn = ((pred == 0) & (tgt == 0)).sum().item()
            fp = ((pred == 1) & (tgt == 0)).sum().item()
            fn = ((pred == 0) & (tgt == 1)).sum().item()

            # accuracy
            acc = (pred == tgt).float().mean().item()

            # balanced accuracy = (TPR + TNR)/2, guard against empty classes
            tpr = tp / (tp + fn) if (tp + fn) > 0 else np.nan
            tnr = tn / (tn + fp) if (tn + fp) > 0 else np.nan
            bal_acc = np.nanmean([tpr, tnr])

            n_obs = int(m.sum().item())
            pos_rate = float(tgt.mean().item())

            # accumulate weighted by n_obs
            bin_stats[feat]["bce_sum"]      += float(bce.item()) * n_obs
            bin_stats[feat]["acc_sum"]      += float(acc) * n_obs
            bin_stats[feat]["bal_acc_sum"]  += float(bal_acc) * n_obs
            bin_stats[feat]["pos_sum"]      += float(pos_rate) * n_obs
            bin_stats[feat]["n_obs"]        += n_obs
            bin_stats[feat]["tp"]           += tp
            bin_stats[feat]["tn"]           += tn
            bin_stats[feat]["fp"]           += fp
            bin_stats[feat]["fn"]           += fn

        # -------------------------
        # Continuous descriptor heads
        # -------------------------
        # cc_pred is (B, len(IDX_CONT)), aligned with IDX_CONT order
        cc_tgt = conds[:, IDX_CONT]
        cc_m   = mask[:, IDX_CONT]

        # per-feature MAE/MSE over observed entries
        for j, feat_idx in enumerate(IDX_CONT):
            feat = FEATURES[feat_idx]
            m = cc_m[:, j].bool()
            if not m.any():
                continue

            diff = (cc_pred[:, j][m] - cc_tgt[:, j][m])
            mae = diff.abs().mean().item()
            mse = (diff.pow(2)).mean().item()

            n_obs = int(m.sum().item())
            obs_rate = float(m.float().mean().item())

            cont_stats[feat]["mae_sum"] += mae * n_obs
            cont_stats[feat]["mse_sum"] += mse * n_obs
            cont_stats[feat]["n_obs"]   += n_obs
            cont_stats[feat]["obs_rate_sum"] += obs_rate * B  # rate averaged per batch; weight by B

    # finalize
    mask_out = {}
    for feat in FEATURES:
        n = mask_stats[feat]["n_sum"]
        mask_out[feat] = {
            "mask_pos_rate": mask_stats[feat]["pos_sum"] / n,
            "mask_bce":      mask_stats[feat]["bce_sum"] / n,
            "mask_acc":      mask_stats[feat]["acc_sum"] / n,
        }

    bin_out = {}
    for feat, st in bin_stats.items():
        n = st["n_obs"]
        if n == 0:
            continue
        # compute confusion-derived metrics on totals (more stable than avg of batch metrics)
        tp, tn, fp, fn = st["tp"], st["tn"], st["fp"], st["fn"]
        tpr = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
        tnr = tn / (tn + fp) if (tn + fp) > 0 else float("nan")
        bal_acc_total = np.nanmean([tpr, tnr])

        bin_out[feat] = {
            "obs_count": int(n),
            "pos_rate":  st["pos_sum"] / n,
            "bce":       st["bce_sum"] / n,
            "acc":       st["acc_sum"] / n,
            "bal_acc":   float(bal_acc_total),
            "tpr":       float(tpr),
            "tnr":       float(tnr),
            "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn)
        }

    cont_out = {}
    for feat, st in cont_stats.items():
        n = st["n_obs"]
        if n == 0:
            continue
        cont_out[feat] = {
            "obs_count": int(n),
            "obs_rate":  st["obs_rate_sum"] / total_n,  # approx overall observed rate
            "mae":       st["mae_sum"] / n,
            "rmse":      float(np.sqrt(st["mse_sum"] / n)),
        }

    return {"mask": mask_out, "binary": bin_out, "continuous": cont_out}


# Example:
aux_detail = evaluate_aux_heads_per_dim(cvae_model, test_loader_ft, device)
print("MASK HEAD PER DIM:", aux_detail["mask"])
print("BINARY HEADS:", aux_detail["binary"])
print("CONT HEADS:", aux_detail["continuous"])


MASK HEAD PER DIM: {'length': {'mask_pos_rate': 1.0, 'mask_bce': 2.191257380349033e-06, 'mask_acc': 1.0}, 'is_assembled': {'mask_pos_rate': 0.30235921830804835, 'mask_bce': 2.503317939033566e-06, 'mask_acc': 1.0}, 'ap': {'mask_pos_rate': 0.3910066855232708, 'mask_bce': 4.854960279790767e-06, 'mask_acc': 1.0}, 'has_beta_sheet_content': {'mask_pos_rate': 0.260606839804577, 'mask_bce': 3.2170216228696218e-06, 'mask_acc': 1.0}, 'hydrophobic_moment': {'mask_pos_rate': 0.260606839804577, 'mask_bce': 1.7991552589104336e-06, 'mask_acc': 1.0}, 'net_charge': {'mask_pos_rate': 0.260606839804577, 'mask_bce': 1.7226151624747653e-06, 'mask_acc': 1.0}}
BINARY HEADS: {'is_assembled': {'obs_count': 9407, 'pos_rate': 0.6771553155465246, 'bce': 1.3347541660966996e-07, 'acc': 1.0, 'bal_acc': 1.0, 'tpr': 1.0, 'tnr': 1.0, 'tp': 6370, 'tn': 3037, 'fp': 0, 'fn': 0}, 'has_beta_sheet_content': {'obs_count': 8108, 'pos_rate': 0.0007400098662095427, 'bce': 5.5316470463473185e-06, 'acc': 1.0, 'bal_acc': 1.0, 'tpr'